In [13]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
import streamlit as st
import numpy as np

In [ ]:
token = "8932168d363cc4fd99402bf906e74083ed57d2261ba8d1b9bb3b4e3be25eb2d9"
url = "https://my.lyfta.app/api/v1/workouts"

# Set the Authorization header with the Bearer prefix
headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.get(url, headers=headers)
data = response.json()

# save json file
with open("/Users/Apple/GitHub/Spotter/data/json_data.json", "w") as f:
    json.dump(data, f, indent=4)

In [15]:
# STEP 1: extract workouts properly
workouts = data["workouts"]

rows = []


# STEP 2: iterate through workouts, exercises, and sets to create a flat structure
for workout in workouts:

    workout_id = workout.get("id")
    date = workout.get("workout_perform_date")
    body_weight = workout.get("body_weight")
    total_volume = workout.get("total_volume")

    for exercise in workout.get("exercises", []):

        exercise_name = exercise.get("excercise_name")
        exercise_type = exercise.get("exercise_type")

        for i, set_data in enumerate(exercise.get("sets", []), start=1):

            weight = set_data.get("weight")
            reps = set_data.get("reps")
            duration = set_data.get("duration")
            distance = set_data.get("distance")

            # clean data
            weight = float(weight) if weight not in ["", None] else None
            reps = float(reps) if reps not in ["", None] else None

            volume = weight * reps if weight is not None and reps is not None else None

            rows.append({
                "workout_id": workout_id,
                "date": date,
                "body_weight": body_weight,
                "exercise": exercise_name,
                "exercise_type": exercise_type,
                "set_number": i,
                "weight": weight,
                "reps": reps,
                "volume": volume,
                "total_workout_volume": total_volume,
                "duration": duration,
                "distance": distance
            })

df = pd.DataFrame(rows)

save_path = "/Users/Apple/GitHub/Spotter/data/workouts.csv"
df.to_csv(save_path, index=False)

print(df.head())

   workout_id                 date  body_weight          exercise  \
0    20160356  2026-05-15 23:00:16         80.0        Cable Curl   
1    20160356  2026-05-15 23:00:16         80.0        Cable Curl   
2    20160356  2026-05-15 23:00:16         80.0        Cable Curl   
3    20160356  2026-05-15 23:00:16         80.0  Run on Treadmill   
4    20160356  2026-05-15 23:00:16         80.0  Run on Treadmill   

       exercise_type  set_number  weight  reps  volume  total_workout_volume  \
0        weight_reps           1    50.0  15.0   750.0                 14130   
1        weight_reps           2    60.0  12.0   720.0                 14130   
2        weight_reps           3    60.0  11.0   660.0                 14130   
3  distance_duration           1     NaN   NaN     NaN                 14130   
4  distance_duration           2     NaN   NaN     NaN                 14130   

  duration distance  
0                    
1                    
2                    
3    10:00     1

In [16]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   workout_id            391 non-null    int64  
 1   date                  391 non-null    object 
 2   body_weight           18 non-null     float64
 3   exercise              391 non-null    object 
 4   exercise_type         391 non-null    object 
 5   set_number            391 non-null    int64  
 6   weight                369 non-null    float64
 7   reps                  371 non-null    float64
 8   volume                369 non-null    float64
 9   total_workout_volume  391 non-null    int64  
 10  duration              36 non-null     object 
 11  distance              36 non-null     object 
dtypes: float64(4), int64(3), object(5)
memory usage: 36.8+ KB
None


In [17]:
# Data Cleaning


print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 391 entries, 0 to 390
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   workout_id            391 non-null    int64  
 1   date                  391 non-null    object 
 2   body_weight           18 non-null     float64
 3   exercise              391 non-null    object 
 4   exercise_type         391 non-null    object 
 5   set_number            391 non-null    int64  
 6   weight                369 non-null    float64
 7   reps                  371 non-null    float64
 8   volume                369 non-null    float64
 9   total_workout_volume  391 non-null    int64  
 10  duration              36 non-null     object 
 11  distance              36 non-null     object 
dtypes: float64(4), int64(3), object(5)
memory usage: 36.8+ KB
None


In [18]:
class FeatureBuilder:
    def __init__(self, df):
        self.df = df.copy()

    def preprocess(self):
        # Convert types
        self.df["date"] = pd.to_datetime(self.df["date"])
        self.df["duration"] = pd.to_numeric(self.df["duration"], errors="coerce")
        self.df["distance"] = pd.to_numeric(self.df["distance"], errors="coerce")

        # Basic derived metric
        self.df["volume"] = self.df["weight"] * self.df["reps"]

        return self.df

    def time_features(self):
        df = self.df

        df = df.sort_values("date")

        df["days_since_prev_workout"] = df.groupby("workout_id")["date"].diff().dt.days
        df["weekday"] = df["date"].dt.weekday
        df["week"] = df["date"].dt.isocalendar().week.astype(int)

        return df

    def exercise_history_features(self):
        df = self.df

        # last weight/reps/volume
        df["prev_weight"] = df.groupby("exercise")["weight"].shift(1)
        df["prev_reps"] = df.groupby("exercise")["reps"].shift(1)
        df["prev_volume"] = df.groupby("exercise")["volume"].shift(1)

        return df

    def rolling_features(self):
        df = self.df

        df["roll3_volume"] = (
            df.groupby("exercise")["volume"]
            .rolling(3, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

        df["roll5_volume"] = (
            df.groupby("exercise")["volume"]
            .rolling(5, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

        return df

    def fatigue_features(self):
        df = self.df

        # 7-day rolling workload (simple fatigue proxy)
        df["daily_volume"] = df.groupby("date")["volume"].transform("sum")

        df["fatigue_7d"] = (
            df.groupby("exercise")["daily_volume"]
            .rolling(7, min_periods=1)
            .sum()
            .reset_index(level=0, drop=True)
        )

        return df

    def progression_features(self):
        df = self.df

        df["weight_delta"] = df["weight"] - df["prev_weight"]
        df["reps_delta"] = df["reps"] - df["prev_reps"]
        df["volume_delta"] = df["volume"] - df["prev_volume"]

        return df

    def build(self):
        self.preprocess()
        self.time_features()
        self.exercise_history_features()
        self.rolling_features()
        self.fatigue_features()
        self.progression_features()

        # clean NaNs
        self.df = self.df.fillna(0)

        return self.df

# RULE-BASED COACH (CORE SYSTEM)

In [19]:
class WorkoutRecommender:

    def __init__(self, df):
        self.df = df

    def muscle_priority_score(self, row):
        """
        Higher score = should be trained next
        """

        fatigue = row["fatigue_7d"]
        recent_volume = row["roll3_volume"]
        days_since = row.get("days_since_prev_workout", 0)

        score = 0

        # recovery rule
        if days_since > 4:
            score += 2

        # undertrained muscle
        if recent_volume < np.percentile(self.df["roll3_volume"], 30):
            score += 2

        # low fatigue = ready to train
        if fatigue < np.percentile(self.df["fatigue_7d"], 50):
            score += 1

        return score

    def recommend_next_exercise(self):

        df = self.df.copy()

        df["priority_score"] = df.apply(self.muscle_priority_score, axis=1)

        # pick top exercise
        top = (
            df.groupby("exercise")["priority_score"]
            .mean()
            .sort_values(ascending=False)
            .head(3)
        )

        return top

#### LOAD PROGRESSION ENGINE (VERY IMPORTANT)

In [20]:
def recommend_next_load(row):

    base_weight = row["weight"]
    fatigue = row["fatigue_7d"]
    progress = row["weight_delta"]

    # simple progression logic
    if fatigue < 10:
        return base_weight * 1.025  # progressive overload

    elif fatigue > 30:
        return base_weight * 0.95   # deload

    else:
        return base_weight

#### ML LAYER (LIGHTWEIGHT ONLY)

In [21]:
from sklearn.ensemble import RandomForestRegressor

def train_model(df):

    features = [
        "prev_weight",
        "prev_reps",
        "roll3_volume",
        "fatigue_7d",
        "weight_delta"
    ]

    target = "weight"

    model = RandomForestRegressor()

    model.fit(df[features], df[target])

    return model